[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/051.%20The%20Classic%20Economic%20Order%20Quantity%20%28EOQ%29%20Problem/P51-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 51. The Classic Economic Order Quantity (EOQ) Problem

## Tier 3: The Advanced Algorithm (Dynamic EOQ with Demand Variability)

### Key Innovations Beyond Classic EOQ
- **Demand Variability**: Handles stochastic demand patterns instead of deterministic
- **Service Level Constraints**: Incorporates target service levels (e.g., 95% fill rate)
- **Safety Stock Optimization**: Automatically calculates optimal safety stock levels
- **Multi-Objective Optimization**: Balances cost minimization with service level maximization
- **Lead Time Considerations**: Accounts for replenishment lead times
- **Particle Swarm Optimization**: Advanced metaheuristic for complex optimization

### Algorithm: PSO-Enhanced Dynamic EOQ

The Particle Swarm Optimization (PSO) approach models each potential EOQ solution as a particle with:
- **Position**: Represents order quantity and reorder point decisions
- **Velocity**: Direction and magnitude of search movement
- **Personal Best**: Best solution found by individual particle
- **Global Best**: Best solution found by entire swarm

### Time Complexity: O(g × p × f) where g = generations, p = population, f = fitness evaluation
### Space Complexity: O(p × d) where p = population, d = problem dimension

### What to Look For in Results
- Robust EOQ solution considering demand uncertainty
- Optimal safety stock levels for target service levels
- Reorder point calculations with lead time considerations
- Multi-objective optimization results
- Convergence analysis showing PSO algorithm performance

In [1]:
from typing import Tuple, List, Dict, Set
from scipy import optimize

# Import required libraries for advanced PSO implementation
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set professional styling for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [2]:
class StochasticDemandGenerator:
    """
    Generate stochastic demand patterns for realistic inventory scenarios.
    """
    
    def __init__(self, base_demand, demand_cv=0.2, seasonality_amplitude=0.1):
        """
        Initialize demand generator with stochastic characteristics.
        
        Parameters:
        - base_demand: Base annual demand rate
        - demand_cv: Coefficient of variation for demand (default 20%)
        - seasonality_amplitude: Seasonal variation amplitude (default 10%)
        """
        self.base_demand = base_demand
        self.demand_cv = demand_cv
        self.seasonality_amplitude = seasonality_amplitude
        
    def generate_daily_demand(self, days=365, seed=None):
        """
        Generate daily demand with stochastic patterns and seasonality.
        """
        if seed is not None:
            np.random.seed(seed)
            
        # Base daily demand
        base_daily = self.base_demand / 365
        
        # Generate seasonal pattern
        seasonal_factor = 1 + self.seasonality_amplitude * np.sin(2 * np.pi * np.arange(days) / 365)
        
        # Generate stochastic demand
        daily_demand = []
        for day in range(days):
            seasonal_demand = base_daily * seasonal_factor[day]
            # Add normal variation
            demand = np.random.normal(seasonal_demand, seasonal_demand * self.demand_cv)
            daily_demand.append(max(0, demand))  # Ensure non-negative demand
            
        return np.array(daily_demand)
    
    def calculate_demand_statistics(self, daily_demand, lead_time_days=7):
        """
        Calculate demand statistics for safety stock calculations.
        """
        # Daily demand statistics
        mean_daily_demand = np.mean(daily_demand)
        std_daily_demand = np.std(daily_demand)
        
        # Lead time demand statistics
        mean_lead_time_demand = mean_daily_demand * lead_time_days
        std_lead_time_demand = std_daily_demand * np.sqrt(lead_time_days)
        
        return {
            'mean_daily_demand': mean_daily_demand,
            'std_daily_demand': std_daily_demand,
            'mean_lead_time_demand': mean_lead_time_demand,
            'std_lead_time_demand': std_lead_time_demand,
            'annual_demand': np.sum(daily_demand)
        }

In [3]:
class Particle:
    """
    Individual particle in the PSO algorithm representing an EOQ solution.
    """
    
    def __init__(self, dimensions, bounds, w=0.7, c1=1.5, c2=1.5):
        """
        Initialize a particle with random position and velocity.
        
        Parameters:
        - dimensions: Number of decision variables (EOQ, reorder_point, review_period)
        - bounds: List of (min, max) tuples for each dimension
        - w: Inertia weight (default 0.7)
        - c1: Cognitive coefficient (default 1.5)
        - c2: Social coefficient (default 1.5)
        """
        self.dimensions = dimensions
        self.bounds = bounds
        self.w = w
        self.c1 = c1
        self.c2 = c2
        
        # Initialize position randomly within bounds
        self.position = np.array([
            np.random.uniform(bounds[i][0], bounds[i][1]) 
            for i in range(dimensions)
        ])
        
        # Initialize velocity randomly
        velocity_range = [(bounds[i][1] - bounds[i][0]) * 0.1 for i in range(dimensions)]
        self.velocity = np.array([
            np.random.uniform(-vr, vr) for vr in velocity_range
        ])
        
        # Personal best position and fitness
        self.personal_best_position = self.position.copy()
        self.personal_best_fitness = float('inf')
        
        # Current fitness
        self.fitness = float('inf')
    
    def update_velocity(self, global_best_position):
        """
        Update particle velocity using PSO equations.
        
        v_new = w * v_old + c1 * r1 * (p_best - x_current) + c2 * r2 * (g_best - x_current)
        """
        r1 = np.random.random(self.dimensions)
        r2 = np.random.random(self.dimensions)
        
        cognitive_component = self.c1 * r1 * (self.personal_best_position - self.position)
        social_component = self.c2 * r2 * (global_best_position - self.position)
        
        self.velocity = self.w * self.velocity + cognitive_component + social_component
        
        # Apply velocity clamping to prevent explosion
        max_velocity = np.array([(self.bounds[i][1] - self.bounds[i][0]) * 0.2 for i in range(self.dimensions)])
        self.velocity = np.clip(self.velocity, -max_velocity, max_velocity)
    
    def update_position(self):
        """
        Update particle position and enforce bounds.
        """
        self.position = self.position + self.velocity
        
        # Enforce bounds
        for i in range(self.dimensions):
            self.position[i] = np.clip(self.position[i], self.bounds[i][0], self.bounds[i][1])
    
    def update_personal_best(self):
        """
        Update personal best if current position is better.
        """
        if self.fitness < self.personal_best_fitness:
            self.personal_best_fitness = self.fitness
            self.personal_best_position = self.position.copy()
            return True
        return False

In [4]:
class PSODynamicEOQ:
    """
    PSO-enhanced Dynamic EOQ optimizer for stochastic inventory management.
    """
    
    def __init__(self, demand_generator, ordering_cost, holding_cost, unit_cost=None, 
                 lead_time_days=7, target_service_level=0.95, shortage_cost=None):
        """
        Initialize PSO Dynamic EOQ optimizer.
        
        Parameters:
        - demand_generator: StochasticDemandGenerator instance
        - ordering_cost: Fixed cost per order
        - holding_cost: Annual holding cost per unit
        - unit_cost: Cost per unit (optional)
        - lead_time_days: Replenishment lead time in days
        - target_service_level: Target fill rate (default 95%)
        - shortage_cost: Cost per unit short (optional, for multi-objective)
        """
        self.demand_gen = demand_generator
        self.S = ordering_cost
        self.H = holding_cost
        self.c = unit_cost
        self.lead_time = lead_time_days
        self.service_level = target_service_level
        self.shortage_cost = shortage_cost
        
        # Generate demand data for optimization
        self.daily_demand = self.demand_gen.generate_daily_demand(days=365, seed=42)
        self.demand_stats = self.demand_gen.calculate_demand_statistics(self.daily_demand, lead_time_days)
        
        # PSO parameters
        self.population_size = 30
        self.max_generations = 100
        self.dimensions = 3  # EOQ, reorder_point, review_period
        
        # Decision variable bounds
        self.bounds = self._calculate_bounds()
        
        # Optimization tracking
        self.convergence_history = []
        self.best_solution = None
        self.best_fitness = float('inf')
        
    def _calculate_bounds(self):
        """
        Calculate reasonable bounds for decision variables.
        """
        # EOQ bounds (based on classic EOQ formula)
        classic_eoq = np.sqrt(2 * self.demand_stats['annual_demand'] * self.S / self.H)
        eoq_bounds = (classic_eoq * 0.3, classic_eoq * 3.0)
        
        # Reorder point bounds (based on lead time demand)
        min_rop = self.demand_stats['mean_lead_time_demand']
        max_rop = self.demand_stats['mean_lead_time_demand'] + 3 * self.demand_stats['std_lead_time_demand']
        rop_bounds = (min_rop, max_rop)
        
        # Review period bounds (1 to 60 days)
        review_bounds = (1, 60)
        
        return [eoq_bounds, rop_bounds, review_bounds]
    
    def calculate_safety_stock(self, service_level):
        """
        Calculate safety stock for target service level.
        """
        z_score = stats.norm.ppf(service_level)
        safety_stock = z_score * self.demand_stats['std_lead_time_demand']
        return safety_stock
    
    def fitness_function(self, position):
        """
        Multi-objective fitness function for EOQ optimization.
        
        Position: [EOQ, reorder_point, review_period]
        """
        eoq, reorder_point, review_period = position
        
        # Basic EOQ costs
        orders_per_year = self.demand_stats['annual_demand'] / eoq
        annual_ordering_cost = orders_per_year * self.S
        annual_holding_cost = (eoq / 2) * self.H
        
        # Safety stock cost
        safety_stock = max(0, reorder_point - self.demand_stats['mean_lead_time_demand'])
        safety_stock_cost = safety_stock * self.H
        
        # Service level penalty (if shortage cost provided)
        service_penalty = 0
        if self.shortage_cost is not None:
            # Estimate service level from reorder point
            if self.demand_stats['std_lead_time_demand'] > 0:
                current_z = (reorder_point - self.demand_stats['mean_lead_time_demand']) / self.demand_stats['std_lead_time_demand']
                current_service_level = stats.norm.cdf(current_z)
                service_shortfall = max(0, self.service_level - current_service_level)
                service_penalty = service_shortfall * self.shortage_cost * self.demand_stats['annual_demand']
        
        # Total cost
        total_cost = annual_ordering_cost + annual_holding_cost + safety_stock_cost + service_penalty
        
        # Add review period cost (more frequent reviews = higher monitoring cost)
        review_cost = (365 / review_period) * 10  # $10 per review
        total_cost += review_cost
        
        return total_cost
    
    def initialize_swarm(self):
        """
        Initialize the particle swarm.
        """
        swarm = []
        for _ in range(self.population_size):
            particle = Particle(self.dimensions, self.bounds)
            particle.fitness = self.fitness_function(particle.position)
            particle.personal_best_fitness = particle.fitness
            swarm.append(particle)
        return swarm
    
    def optimize(self, verbose=True):
        """
        Run PSO optimization.
        """
        # Initialize swarm
        swarm = self.initialize_swarm()
        
        # Find initial global best
        global_best_particle = min(swarm, key=lambda p: p.fitness)
        global_best_position = global_best_particle.position.copy()
        global_best_fitness = global_best_particle.fitness
        
        # Optimization loop
        for generation in range(self.max_generations):
            generation_best_fitness = float('inf')
            
            for particle in swarm:
                # Update velocity and position
                particle.update_velocity(global_best_position)
                particle.update_position()
                
                # Evaluate fitness
                particle.fitness = self.fitness_function(particle.position)
                
                # Update personal best
                particle.update_personal_best()
                
                # Track generation best
                if particle.fitness < generation_best_fitness:
                    generation_best_fitness = particle.fitness
            
            # Update global best
            current_best_particle = min(swarm, key=lambda p: p.fitness)
            if current_best_particle.fitness < global_best_fitness:
                global_best_fitness = current_best_particle.fitness
                global_best_position = current_best_particle.position.copy()
            
            # Record convergence
            self.convergence_history.append(global_best_fitness)
            
            # Verbose output
            if verbose and (generation % 10 == 0 or generation == self.max_generations - 1):
                print(f"Generation {generation:3d}: Best Fitness = {global_best_fitness:,.2f}")
        
        # Store final results
        self.best_solution = global_best_position
        self.best_fitness = global_best_fitness
        
        return global_best_position, global_best_fitness
    
    def analyze_solution(self, position=None):
        """
        Analyze the optimal solution and provide detailed insights.
        """
        if position is None:
            position = self.best_solution
        
        eoq, reorder_point, review_period = position
        
        # Calculate detailed metrics
        orders_per_year = self.demand_stats['annual_demand'] / eoq
        days_between_orders = 365 / orders_per_year
        avg_inventory = eoq / 2
        
        # Safety stock analysis
        safety_stock = max(0, reorder_point - self.demand_stats['mean_lead_time_demand'])
        target_safety_stock = self.calculate_safety_stock(self.service_level)
        
        # Service level estimation
        if self.demand_stats['std_lead_time_demand'] > 0:
            current_z = (reorder_point - self.demand_stats['mean_lead_time_demand']) / self.demand_stats['std_lead_time_demand']
            estimated_service_level = stats.norm.cdf(current_z)
        else:
            estimated_service_level = 1.0
        
        # Cost breakdown
        annual_ordering_cost = orders_per_year * self.S
        annual_holding_cost = (avg_inventory + safety_stock) * self.H
        review_cost = (365 / review_period) * 10
        
        total_inventory_cost = annual_ordering_cost + annual_holding_cost + review_cost
        
        # Compare with classic EOQ
        classic_eoq = np.sqrt(2 * self.demand_stats['annual_demand'] * self.S / self.H)
        classic_metrics = {
            'eoq': classic_eoq,
            'annual_ordering_cost': (self.demand_stats['annual_demand'] / classic_eoq) * self.S,
            'annual_holding_cost': (classic_eoq / 2) * self.H,
            'total_cost': (self.demand_stats['annual_demand'] / classic_eoq) * self.S + (classic_eoq / 2) * self.H
        }
        
        return {
            'optimal_eoq': eoq,
            'optimal_reorder_point': reorder_point,
            'optimal_review_period': review_period,
            'orders_per_year': orders_per_year,
            'days_between_orders': days_between_orders,
            'average_inventory': avg_inventory,
            'safety_stock': safety_stock,
            'target_safety_stock': target_safety_stock,
            'estimated_service_level': estimated_service_level,
            'target_service_level': self.service_level,
            'annual_ordering_cost': annual_ordering_cost,
            'annual_holding_cost': annual_holding_cost,
            'review_cost': review_cost,
            'total_inventory_cost': total_inventory_cost,
            'fitness_value': self.best_fitness,
            'classic_comparison': classic_metrics,
            'cost_increase_vs_classic': ((total_inventory_cost - classic_metrics['total_cost']) / classic_metrics['total_cost']) * 100
        }

In [5]:
# Visualization functions for PSO Dynamic EOQ

def plot_pso_convergence(convergence_history):
    """
    Plot PSO convergence history.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    generations = range(len(convergence_history))
    ax.plot(generations, convergence_history, 'b-', linewidth=2.5, alpha=0.8)
    ax.scatter(generations[-1], convergence_history[-1], color='red', s=100, zorder=5, 
               label=f'Final: {convergence_history[-1]:,.2f}')
    
    ax.set_xlabel('Generation', fontsize=14, fontweight='bold')
    ax.set_ylabel('Best Fitness (Total Cost)', fontsize=14, fontweight='bold')
    ax.set_title('PSO Convergence History', fontsize=16, fontweight='bold', pad=20)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Format y-axis as currency
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    plt.tight_layout()
    plt.show()
    
    return fig

def plot_solution_analysis(analysis_results, demand_stats):
    """
    Create comprehensive solution analysis visualization.
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Panel 1: Cost Comparison
    classic = analysis_results['classic_comparison']
    dynamic = analysis_results
    
    categories = ['Ordering Cost', 'Holding Cost', 'Total Cost']
    classic_costs = [classic['annual_ordering_cost'], classic['annual_holding_cost'], classic['total_cost']]
    dynamic_costs = [dynamic['annual_ordering_cost'], dynamic['annual_holding_cost'], dynamic['total_inventory_cost']]
    
    x = np.arange(len(categories))
    width = 0.35
    
    ax1.bar(x - width/2, classic_costs, width, label='Classic EOQ', alpha=0.8, color='lightblue')
    ax1.bar(x + width/2, dynamic_costs, width, label='Dynamic EOQ', alpha=0.8, color='lightcoral')
    
    ax1.set_xlabel('Cost Component', fontweight='bold')
    ax1.set_ylabel('Annual Cost ($)', fontweight='bold')
    ax1.set_title('Cost Comparison: Classic vs Dynamic EOQ', fontweight='bold', fontsize=14)
    ax1.set_xticks(x)
    ax1.set_xticklabels(categories)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # Panel 2: Service Level Analysis
    service_levels = [analysis_results['target_service_level'], analysis_results['estimated_service_level']]
    service_labels = ['Target Service Level', 'Estimated Service Level']
    colors = ['green', 'blue']
    
    bars = ax2.bar(service_labels, service_levels, color=colors, alpha=0.7)
    ax2.set_ylabel('Service Level', fontweight='bold')
    ax2.set_title('Service Level Achievement', fontweight='bold', fontsize=14)
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels
    for bar, level in zip(bars, service_levels):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{level:.1%}', ha='center', va='bottom', fontweight='bold')
    
    # Panel 3: Safety Stock Analysis
    safety_stocks = [analysis_results['target_safety_stock'], analysis_results['safety_stock']]
    safety_labels = ['Target Safety Stock', 'Optimal Safety Stock']
    
    ax3.bar(safety_labels, safety_stocks, color=['orange', 'purple'], alpha=0.7)
    ax3.set_ylabel('Safety Stock (units)', fontweight='bold')
    ax3.set_title('Safety Stock Analysis', fontweight='bold', fontsize=14)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Panel 4: Demand Pattern Visualization
    # Generate sample demand pattern for visualization
    sample_days = 60
    sample_demand = demand_stats['mean_daily_demand'] + np.random.normal(0, demand_stats['std_daily_demand'], sample_days)
    sample_demand = np.maximum(sample_demand, 0)  # Ensure non-negative
    
    days = np.arange(1, sample_days + 1)
    ax4.plot(days, sample_demand, 'b-', alpha=0.6, linewidth=1.5)
    ax4.axhline(y=demand_stats['mean_daily_demand'], color='red', linestyle='--', 
                label=f'Mean: {demand_stats["mean_daily_demand"]:.1f}', linewidth=2)
    ax4.fill_between(days, demand_stats['mean_daily_demand'], sample_demand, 
                     where=(sample_demand > demand_stats['mean_daily_demand']), 
                     alpha=0.3, color='red', label='Above Mean')
    ax4.fill_between(days, demand_stats['mean_daily_demand'], sample_demand, 
                     where=(sample_demand <= demand_stats['mean_daily_demand']), 
                     alpha=0.3, color='blue', label='Below Mean')
    
    ax4.set_xlabel('Day', fontweight='bold')
    ax4.set_ylabel('Daily Demand (units)', fontweight='bold')
    ax4.set_title('Stochastic Demand Pattern (Sample)', fontweight='bold', fontsize=14)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return fig

### Concrete Example - MegaMart Coffee Beans with Demand Variability

Let's apply the PSO-enhanced Dynamic EOQ to MegaMart's Colombian Supremo coffee bean problem with realistic demand variability:

**Enhanced Problem Parameters:**
- Annual demand (D): 12,000 bags per year (base rate)
- Demand variability: 20% coefficient of variation
- Seasonality: 10% seasonal amplitude
- Ordering cost (S): $150 per order
- Holding cost (H): $8 per bag per year
- Lead time: 7 days
- Target service level: 95%
- Shortage cost: $50 per bag (for multi-objective optimization)

In [ ]:
try:
    try:
        # Initialize demand generator with realistic variability
        demand_gen = StochasticDemandGenerator(
            base_demand=12000,      # 12,000 bags per year
            demand_cv=0.2,         # 20% coefficient of variation
            seasonality_amplitude=0.1  # 10% seasonal variation
        )
        
        # Initialize PSO Dynamic EOQ optimizer
        pso_optimizer = PSODynamicEOQ(
            demand_generator=demand_gen,
            ordering_cost=150,      # $150 per order
            holding_cost=8,        # $8 per bag per year
            unit_cost=25,          # $25 per bag
            lead_time_days=7,      # 7 days lead time
            target_service_level=0.95,  # 95% service level
            shortage_cost=50      # $50 per bag shortage
        )
        
        # Display demand statistics
        print("=== Demand Statistics ===")
        print(f"Mean Daily Demand: {pso_optimizer.demand_stats['mean_daily_demand']:.2f} bags/day")
        print(f"Std Daily Demand: {pso_optimizer.demand_stats['std_daily_demand']:.2f} bags/day")
        print(f"Mean Lead Time Demand: {pso_optimizer.demand_stats['mean_lead_time_demand']:.2f} bags")
        print(f"Std Lead Time Demand: {pso_optimizer.demand_stats['std_lead_time_demand']:.2f} bags")
        print(f"Annual Demand: {pso_optimizer.demand_stats['annual_demand']:.0f} bags")
        
        # Calculate target safety stock
        target_safety_stock = pso_optimizer.calculate_safety_stock(0.95)
        print(f"\nTarget Safety Stock (95% SL): {target_safety_stock:.2f} bags")
    except Exception as e:
        print(f'Error in cell: {e}')
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    # Run PSO optimization
    print("\n=== PSO Dynamic EOQ Optimization ===")
    print("Optimizing dynamic EOQ with PSO...")
    
    optimal_position, optimal_fitness = pso_optimizer.optimize(verbose=True)
    
    print(f"\nOptimization completed!")
    print(f"Converged at iteration {len(pso_optimizer.convergence_history)}")
    print(f"Final Fitness Value: {optimal_fitness:,.2f}")
except Exception as e:
    print(f'Error in cell: {e}')


=== PSO Dynamic EOQ Optimization ===
Optimizing dynamic EOQ with PSO...
Error in cell: name 'pso_optimizer' is not defined


In [ ]:
try:
    # Analyze the optimal solution
    analysis = pso_optimizer.analyze_solution()
    
    print("\n=== MegaMart Dynamic EOQ with PSO Optimization ===")
    print(f"Optimal Order Quantity: {analysis['optimal_eoq']:.2f} bags")
    print(f"Optimal Reorder Point: {analysis['optimal_reorder_point']:.2f} bags")
    print(f"Optimal Review Period: {analysis['optimal_review_period']:.1f} days")
    print(f"Optimization Fitness: {analysis['fitness_value']:.2f}")
    print(f"\nRecommended Safety Stock: {analysis['safety_stock']:.2f} bags")
    print(f"Target Safety Stock: {analysis['target_safety_stock']:.2f} bags")
    print(f"Estimated Service Level: {analysis['estimated_service_level']:.1%}")
    print(f"Target Service Level: {analysis['target_service_level']:.1%}")
    
    print("\n=== Operational Metrics ===")
    print(f"Orders per Year: {analysis['orders_per_year']:.2f}")
    print(f"Days Between Orders: {analysis['days_between_orders']:.1f}")
    print(f"Average Inventory: {analysis['average_inventory']:.2f} bags")
    
    print("\n=== Cost Breakdown ===")
    print(f"Annual Ordering Cost: ${analysis['annual_ordering_cost']:,.2f}")
    print(f"Annual Holding Cost: ${analysis['annual_holding_cost']:,.2f}")
    print(f"Review Cost: ${analysis['review_cost']:,.2f}")
    print(f"Total Inventory Cost: ${analysis['total_inventory_cost']:,.2f}")
    
    print("\n=== Comparison with Classic EOQ ===")
    classic = analysis['classic_comparison']
    print(f"Classic EOQ: {classic['eoq']:.2f} bags")
    print(f"Classic Total Cost: ${classic['total_cost']:,.2f}")
    print(f"Dynamic EOQ Cost Increase: {analysis['cost_increase_vs_classic']:+.2f}%")
except Exception as e:
    print(f'Error in cell: {e}')

Error in cell: name 'pso_optimizer' is not defined


In [ ]:
try:
    # Plot PSO convergence history
    fig1 = plot_pso_convergence(pso_optimizer.convergence_history)
    plt.show()
    
    # Plot comprehensive solution analysis
    fig2 = plot_solution_analysis(analysis, pso_optimizer.demand_stats)
    plt.show()
except Exception as e:
    print(f'Error in cell: {e}')

Error in cell: name 'pso_optimizer' is not defined


### Why This Advanced PSO Approach vs Classic EOQ

The PSO-enhanced Dynamic EOQ provides significant advantages over both mathematical and heuristic approaches:

#### Key Advantages:
1. **Demand Uncertainty Handling**: Incorporates stochastic demand patterns and variability
2. **Service Level Optimization**: Achieves target service levels with safety stock calculations
3. **Multi-Objective Optimization**: Balances cost minimization with service level maximization
4. **Lead Time Considerations**: Accounts for replenishment lead times in reorder point calculations
5. **Robust Solutions**: PSO finds globally optimal solutions in complex search spaces
6. **Adaptive Review Periods**: Optimizes not just EOQ but also review frequency

#### Advanced Features:
- **Particle Swarm Intelligence**: Collective intelligence for complex optimization
- **Convergence Tracking**: Monitors optimization progress and solution quality
- **Seasonal Demand Handling**: Incorporates demand seasonality and trends
- **Risk Management**: Explicit service level constraints and shortage cost considerations

#### When to Use This Advanced Approach:
- **High-Value Items**: When stockouts are very costly and service levels are critical
- **Volatile Demand**: When demand patterns are unpredictable or highly variable
- **Multi-Echelon Systems**: When coordinating inventory across multiple locations
- **Strategic Planning**: When making long-term inventory policy decisions
- **Complex Supply Chains**: When lead times are significant and demand is stochastic

#### Trade-offs:
- **Computational Complexity**: Requires more processing power than simple formulas
- **Parameter Tuning**: PSO parameters need careful calibration
- **Data Requirements**: Needs demand history and statistical analysis

### Key Insights from Advanced Analysis

The PSO Dynamic EOQ analysis reveals critical insights for modern inventory management:

1. **Safety Stock Importance**: With demand variability, safety stock becomes crucial for service level achievement
2. **Cost-Service Trade-off**: Higher service levels require additional safety stock investment
3. **Review Period Optimization**: Frequent reviews can reduce safety stock requirements
4. **Demand Pattern Impact**: Seasonality and variability significantly affect optimal policies
5. **Robust Solutions**: PSO finds solutions that perform well under uncertainty

This advanced approach represents the state-of-the-art in inventory optimization, bridging the gap between classical EOQ theory and modern supply chain complexity. It provides the robust, service-oriented solutions needed in today's volatile business environment.